# Chạy thử 3.2 (CIFAR-10, PreAct ResNet-18) trên Kaggle

Mục đích: đo **thời gian mỗi epoch** trên GPU thật, kiểm tra code chạy được trên Kaggle
và kiểm tra **resume**. Tổng thời gian khoảng 10–20 phút GPU.

**Trước khi chạy** (xem `kaggle/README.md`):
1. Settings → Accelerator: **GPU P100** (bài báo dùng P100) hoặc GPU T4 x2.
2. Settings → **Internet: On** (để tải CIFAR-10).
3. Add Input → dataset chứa thư mục `Source` của nhóm.

In [ ]:
# ==== Tham số ====
SMOKE_EPOCHS = 2          # số epoch chạy thử cho mỗi cấu hình
RUN_FULL = False          # True: chạy thật 200 epoch sau khi chạy thử (≈ vài giờ)
FULL_ALPHA = 1.0          # 0 = ERM, 1 = mixup như bài báo
FULL_SEED = 0
WORKERS = 4               # Kaggle có 4 nhân CPU

In [ ]:
import os, sys, shutil, glob, subprocess, json, time
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CHƯA BẬT GPU: Settings -> Accelerator -> GPU")
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Tìm thư mục Source trong /kaggle/input (thư mục có chứa core/ và cifar/)
cands = [os.path.dirname(os.path.dirname(p)) for p in glob.glob('/kaggle/input/**/core/mixup.py', recursive=True)]
assert cands, "Không thấy Source. Add Input -> chọn dataset chứa thư mục Source"
SRC_IN = cands[0]
SRC = '/kaggle/working/Source'
if not os.path.exists(SRC):
    shutil.copytree(SRC_IN, SRC)
os.chdir(SRC)
DATA = '/kaggle/working/data'
OUT = '/kaggle/working/results'
print("Source:", SRC_IN, "->", SRC)

In [ ]:
# Test phần lõi (vài giây). Kaggle thường đã có pytest; nếu chưa thì dùng cách chạy trực tiếp.
!python -m pytest tests -q || python tests/test_core.py

## 1. Đo thời gian mỗi epoch: FP32 và AMP

Chạy mixup α=1 trên CIFAR-10 đầy đủ trong `SMOKE_EPOCHS` epoch, một lần FP32 (giống bài báo)
và một lần bật `--amp` (mixed precision, thường nhanh hơn trên T4/P100).

In [ ]:
def run(args):
    cmd = f"python experiments/exp32_cifar.py --data-dir {DATA} --out-dir {OUT} --workers {WORKERS} " + args
    print("$", cmd, flush=True)
    t = time.time()
    # Đọc từng dòng để log hiện ngay dưới cell (subprocess.run thường chỉ ghi vào log của Kaggle)
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for line in proc.stdout:
        print(line, end="", flush=True)
    rc = proc.wait()
    print(f"-> xong sau {time.time()-t:.0f}s, exit={rc}", flush=True)
    assert rc == 0, "Lệnh lỗi, xem log phía trên"

run(f"--alpha 1 --seed 0 --epochs {SMOKE_EPOCHS} --run-name smoke/fp32")
run(f"--alpha 1 --seed 0 --epochs {SMOKE_EPOCHS} --run-name smoke/amp --amp")

## 2. Kiểm tra resume

Chạy lại `smoke/fp32` với số epoch lớn hơn 1. Log phải có dòng
`[resume] tiếp tục từ epoch N` và chỉ chạy thêm đúng 1 epoch.

In [ ]:
run(f"--alpha 1 --seed 0 --epochs {SMOKE_EPOCHS + 1} --run-name smoke/fp32")
import pandas as pd
h = pd.read_csv(f"{OUT}/smoke/fp32/history.csv")
display(h)
assert list(h.epoch) == list(range(SMOKE_EPOCHS + 1)), "Resume sai: số epoch trong log không khớp"
print("Resume OK")

## 3. Ước lượng thời gian cho các lần chạy thật

Bỏ epoch đầu (có thời gian khởi động cuDNN, nạp dữ liệu) khi tính trung bình nếu có thể.
Con số dưới đây là **ước lượng**, nên ghi lại kèm tên GPU vào báo cáo/kế hoạch.

In [ ]:
rows = []
for mode in ["fp32", "amp"]:
    h = pd.read_csv(f"{OUT}/smoke/{mode}/history.csv")
    t = h.epoch_time_s.iloc[1:].mean() if len(h) > 1 else h.epoch_time_s.mean()
    rows.append({"mode": mode, "giây/epoch": round(t, 1),
                 "200 epoch (giờ)": round(t * 200 / 3600, 2),
                 "100 epoch (giờ)": round(t * 100 / 3600, 2)})
est = pd.DataFrame(rows)
est["gpu"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
display(est)
est.to_csv(f"{OUT}/smoke/timing_estimate.csv", index=False)

## 4. (Tùy chọn) Chạy thật 200 epoch

Đặt `RUN_FULL = True` ở ô đầu rồi chạy lại. Nên dùng **Save Version → Save & Run All (Commit)**
để Kaggle chạy nền tối đa 12 giờ, không cần mở trình duyệt.

In [ ]:
if RUN_FULL:
    name = ("erm" if FULL_ALPHA <= 0 else f"mixup_a{FULL_ALPHA:g}") + f"_seed{FULL_SEED}"
    run(f"--alpha {FULL_ALPHA} --seed {FULL_SEED} --epochs 200 --run-name exp32_cifar10/{name}")
    print(open(f"{OUT}/exp32_cifar10/{name}/summary.json").read())
else:
    print("Bỏ qua chạy thật (RUN_FULL = False)")

In [ ]:
# Đóng gói kết quả (không kèm file .pt cho nhẹ) để tải về từ tab Output
!cd /kaggle/working && zip -qr results_logs.zip results -x '*.pt' && ls -lh results_logs.zip